# Structured RAG

In [65]:
from openai import OpenAI
from dotenv import load_dotenv
from typing import Optional, Literal
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index
from pydantic import BaseModel, Field, model_validator
import json
import os

## RAG

In [4]:
reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]

chunked_docs = chunk_documents(
    parsed_docs,
    size=3000,
    step=1500
)

index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")

Indexed 385 chunks from 95 documents


In [5]:
def search(index, query):
    return index.search(
        query=query,
        num_results=5,
    )

In [7]:
instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

In [21]:
def augment(template, question, search_results):
    context = json.dumps(search_results, indent=2)
    return template.format(
        question=question, 
        context=context
    )

In [24]:
def llm(client, user_prompt, system_instructions=None, model="gpt-4o-mini"):
    messages = []

    if system_instructions:
        messages.append({
            "role":"system",
            "content":system_instructions
        })
    
    messages.append({
        "role":"user",
        "content":user_prompt
    })

    response = client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [23]:
def rag(client, user_template, index, query, system_prompt=None, model="gpt-4o-mini"):
    search_results = search(index, query)
    prompt = augment(user_template, query, search_results)
    return llm(client, prompt, system_instructions=system_prompt, model=model)

In [15]:
load_dotenv()
openai_key = os.environ.get("OPENAI_API_KEY")

In [25]:
openai_client = OpenAI(api_key=openai_key)
query = "How do I make implement an LLM as a judge?"
print(rag(openai_client, prompt_template, index, query, instructions))

To implement an LLM as a judge, follow these steps based on the provided context:

1. **Install the Necessary Libraries**:
   - Use the command: 
     ```python
     pip install evidently
     ```

2. **Import Required Modules**:
   - Include the necessary modules in your Python script:
     ```python
     import pandas as pd
     import numpy as np
     from evidently import Dataset, DataDefinition, Report, BinaryClassification
     from evidently.descriptors import *
     from evidently.presets import TextEvals, ClassificationPreset
     from evidently.llm.templates import BinaryClassificationPromptTemplate
     ```

3. **Set Up Your OpenAI API Key**:
   - Pass your OpenAI API key as an environment variable:
     ```python
     import os
     os.environ["OPENAI_API_KEY"] = "YOUR_KEY"
     ```

4. **Create Your Evaluation Dataset**:
   - Set up a toy Q&A dataset containing questions, target responses (approved), new responses (imitated), and manual labels indicating correctness.

5. *

## Structured LLM Response Example

In [31]:
def structured_llm(client, user_prompt, output_type, system_instructions=None, model="gpt-4o-mini"):
    messages = []

    if system_instructions:
        messages.append({
            "role":"system",
            "content":system_instructions
        })
    
    messages.append({
        "role":"user",
        "content":user_prompt
    })

    response = client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )

    return response.output_parsed

In [29]:
class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

In [32]:
response = structured_llm(
    openai_client, 
    user_prompt="Alice and Bob are going to a science fair on Friday",
    output_type=CalendarEvent,
    system_instructions="Extract the event information"
)
response

CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

## Basic Structured RAG 

In [43]:
class RAGResponse(BaseModel):
    query: str
    answer: Optional[str] = None
    found_answer: bool


In [35]:
RAGResponse.model_json_schema()

{'properties': {'query': {'title': 'Query', 'type': 'string'},
  'answer': {'title': 'Answer', 'type': 'string'},
  'found_answer': {'title': 'Found Answer', 'type': 'boolean'}},
 'required': ['query', 'answer', 'found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [38]:
def structured_rag(client, user_template, index, query, output_format, 
system_prompt=None, model="gpt-4o-mini"):
    search_results = search(index, query)
    prompt = augment(user_template, query, search_results)
    return structured_llm(client, prompt, output_format, 
    system_instructions=system_prompt, model=model
    )

In [40]:
structured_response = structured_rag(
    openai_client,
    prompt_template,
    index,
    query,
    RAGResponse,
    instructions,
)
print(structured_response)

query='How do I make implement an LLM as a judge?' answer='To implement an LLM as a judge, follow these steps:\n\n1. **Install the Required Library**: Use the command `pip install evidently` to install the Evidently library.\n\n2. **Import Necessary Modules**: Include the required modules in your Python script:\n   ```python\n   import pandas as pd\n   import numpy as np\n   from evidently import Dataset, DataDefinition, Report, BinaryClassification\n   from evidently.llm.templates import BinaryClassificationPromptTemplate\n   ```\n\n3. **Set Up the OpenAI API Key**: Pass your OpenAI API key as an environment variable:\n   ```python\n   import os\n   os.environ["OPENAI_API_KEY"] = "YOUR_KEY"\n   ```\n\n4. **Create Evaluation Dataset**: Develop a dataset with questions, target responses, new responses, and manual labels to help evaluate the judge\'s performance.\n\n5. **Design the LLM Evaluator Prompt**: Create a prompt template to evaluate the responses based on desired criteria:\n   `

In [41]:
new_query = "How do I dual-boot Ubuntu on a Windows machine?"
structured_response = structured_rag(
    openai_client,
    prompt_template,
    index,
    new_query,
    RAGResponse,
    instructions,
)
print(structured_response.answer[:200])
print(structured_response.found_answer)

The context does not provide any information on dual-booting Ubuntu on a Windows machine.
False


In [44]:
new_instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.

If you don't find the answer, set `answer` to None
"""

In [46]:
new_query = "How do I dual-boot Ubuntu on a Windows machine?"
structured_response = structured_rag(
    openai_client,
    prompt_template,
    index,
    new_query,
    RAGResponse,
    new_instructions,
)
print(structured_response.answer)
print(structured_response.found_answer)

None
False


## Adding Docstrings and Field Descriptions to the Ouput Class

In [75]:
class DocAssistantResponse(BaseModel):
    """
    The response from the documentation RAG system.
    """
    query: str = Field(description="The question the user provided to be answered.")
    answer: Optional[str] = Field(None, description="Answer to the query or None if it's not found")
    found_answer: bool = Field(description="True if the answer to the query is found, False otherwise")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask"
    )

In [76]:
DocAssistantResponse.model_json_schema()

{'description': 'The response from the documentation RAG system.',
 'properties': {'query': {'description': 'The question the user provided to be answered.',
   'title': 'Query',
   'type': 'string'},
  'answer': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'description': "Answer to the query or None if it's not found",
   'title': 'Answer'},
  'found_answer': {'description': 'True if the answer to the query is found, False otherwise',
   'title': 'Found Answer',
   'type': 'boolean'},
  'confidence': {'description': 'Confidence score from 0.0 to 1.0 indicating how certain the answer is',
   'title': 'Confidence',
   'type': 'number'},
  'confidence_explanation': {'description': 'Explanation about the confidence level',
   'title': 'Confidence Explanation',
   'type': 'string'},
  'answer_type': {'description': 'The category of the answer',
   'enum': ['how-to',
    'explanation',
    'troubleshooting',
    'comparison',
    'reference'],
   'title': 'Answer

In [77]:
answer = structured_rag(
    openai_client,
    prompt_template,
    index,
    query,
    DocAssistantResponse
)

In [78]:
answer.query

'How do I make implement an LLM as a judge?'

In [79]:
answer.answer

'To implement an LLM as a judge, follow these steps:\n1. **Install Required Libraries**:\n   - Use `pip install evidently` for the Evidently library.\n   - Import necessary modules like `pandas`, `evidently.Dataset`, etc.\n\n2. **Set Up OpenAI API Key**:\n   - Set your OpenAI API key in the environment:\n     ```python\n     import os\n     os.environ["OPENAI_API_KEY"] = "YOUR_KEY"\n     ```\n\n3. **Create Evaluation Dataset**:\n   - Construct a toy Q&A dataset that includes:\n     - Questions sent to the LLM.\n     - Approved target responses.\n     - New responses from the system.\n     - Manual labels for correctness.\n   - Example structure:\n     ```python\n     data = [\n         ["Question?", "Target response", "New response", "label", "explanation"],\n         ...\n     ]\n     ```\n\n4. **Design the LLM Evaluator Prompt**:\n   - Create a Binary Classification Prompt Template that defines criteria for correct/incorrect responses. For example:\n     ```python\n     correctness =

In [80]:
print(answer.confidence)
answer.confidence_explanation

0.95


'The answer is based on comprehensive steps provided in the context about how to implement an LLM as a judge, covering installation, setup, dataset creation, and usage of the Evidently framework.'

In [81]:
answer.answer_type

'how-to'

In [82]:
answer.followup_questions

['What kind of datasets should I use for evaluation?',
 'How can I refine the LLM judge after initial testing?',
 'Are there specific examples of prompts for different judging criteria?']

In [83]:
new_query = "How do I dual-boot Ubuntu on a Windows machine?"
structured_response = structured_rag(
    openai_client,
    prompt_template,
    index,
    new_query,
    DocAssistantResponse,
)
structured_response

DocAssistantResponse(query='How do I dual-boot Ubuntu on a Windows machine?', answer=None, found_answer=False, confidence=0.0, confidence_explanation='No relevant information found in the provided context regarding dual-booting Ubuntu with Windows.', answer_type='reference', followup_questions=['What are the steps to install Ubuntu alongside Windows?', 'What tools do I need for dual-booting?', 'Can I dual-boot with UEFI or Legacy BIOS?'])

## Nested Fields with Mutual Exclusivity

In [111]:
class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

class AnswerNotFound(BaseModel):
    explanation: str

class AnswerResponse(BaseModel):
    """
    If answer is found, 'answer' is populated.
    If no answer is found, 'answer_not_found' is populated.
    Only one of the two fields can be set at a time. Never both or neither.
    """

    answer_not_found: Optional[AnswerNotFound] = None
    found_answer: bool
    answer: Optional[RAGResponse] = None

    @model_validator(mode="after")
    def check_consistency(self):
        if self.answer is not None and self.answer_not_found is not None:
            raise ValueError("Provide either 'answer' or 'answer_not_found', not both.")

        if self.answer is None and self.answer_not_found is None:
            raise ValueError("Provide either 'answer' or 'answer_not_found'.")

        return self

In [112]:
AnswerResponse.model_json_schema()

{'$defs': {'AnswerNotFound': {'properties': {'explanation': {'title': 'Explanation',
     'type': 'string'}},
   'required': ['explanation'],
   'title': 'AnswerNotFound',
   'type': 'object'},
  'RAGResponse': {'description': 'This model provides a structured answer with metadata about the response,\nincluding confidence, categorization, and follow-up suggestions.',
   'properties': {'answer': {'description': "The main answer to the user's question in markdown",
     'title': 'Answer',
     'type': 'string'},
    'found_answer': {'description': 'True if relevant information was found in the documentation',
     'title': 'Found Answer',
     'type': 'boolean'},
    'confidence': {'description': 'Confidence score from 0.0 to 1.0 indicating how certain the answer is',
     'title': 'Confidence',
     'type': 'number'},
    'confidence_explanation': {'description': 'Explanation about the confidence level',
     'title': 'Confidence Explanation',
     'type': 'string'},
    'answer_type': 

In [116]:
new_query = "how do I dual boot Ubuntu on windows?"
structured_response = structured_rag(
    openai_client,
    prompt_template,
    index,
    new_query,
    AnswerResponse,
    model="gpt-5.2-2025-12-11"
)
structured_response

AnswerResponse(answer_not_found=None, found_answer=False, answer=RAGResponse(answer='Below is the standard way to dual‑boot Ubuntu alongside Windows (UEFI systems). If you tell me your PC model and whether Windows is installed in **UEFI** mode, I can tailor it.\n\n## 1) Prep in Windows (important)\n1. **Back up** anything important.\n2. **Disable Fast Startup**: Control Panel → Power Options → “Choose what the power buttons do” → uncheck **Turn on fast startup**.\n3. **Check disk type (UEFI/GPT)**:\n   - Press `Win+X` → Disk Management → right‑click Disk 0 → Properties → Volumes.\n   - If it says **GPT**, you’re on UEFI (good/typical).\n4. **Shrink the Windows partition** to make space:\n   - Disk Management → right‑click `C:` → **Shrink Volume**.\n   - Give Ubuntu at least **25–50 GB** (more if you can).\n   - Leave the space **Unallocated** (don’t create a new Windows volume).\n5. (Optional but common) **Suspend BitLocker** if enabled (it can interfere with boot changes): Windows Sec

In [114]:
new_query = "how do I dual boot Ubuntu on windows?"
structured_response = structured_rag(
    openai_client,
    prompt_template,
    index,
    new_query,
    AnswerResponse,
    model="gpt-5-2025-08-07"
)
structured_response

AnswerResponse(answer_not_found=AnswerNotFound(explanation='The provided documentation is about Evidently’s evaluation platform and does not include instructions for dual-booting Ubuntu with Windows. Please consult Ubuntu’s official installation guide or ask a question related to the provided docs.'), found_answer=False, answer=None)

In [115]:
new_query = "how do I dual boot Ubuntu on windows?"
structured_response = structured_rag(
    openai_client,
    prompt_template,
    index,
    new_query,
    AnswerResponse,
    model="gpt-5-mini-2025-08-07"
)
structured_response

AnswerResponse(answer_not_found=None, found_answer=True, answer=RAGResponse(answer='Short version\n\n1) Back up your important Windows data.\n2) Turn off Windows Fast Startup and (if used) suspend BitLocker.\n3) Free up space on your disk using Windows Disk Management (shrink a partition, leave at least 20–50 GB for Ubuntu).\n4) Download the Ubuntu ISO and create a bootable USB (Rufus or balenaEtcher). Use GPT/FAT32 for UEFI or MBR/NTFS for legacy BIOS as appropriate.\n5) Boot the PC from the USB in the same mode Windows uses (UEFI vs Legacy).\n6) Run the Ubuntu installer and choose “Install Ubuntu alongside Windows” or use “Something else” for manual partitioning. Install the bootloader to the EFI partition (or disk for legacy).\n7) Finish installation, reboot, and choose Ubuntu or Windows from the GRUB menu.\n\nDetailed step-by-step guide\n\nPreparation and safety\n\n- Back up everything important from Windows before you begin. Dual-booting usually works fine, but mistakes or disk pr

**How do you make sure that the answer is relevant to the context provided when the question is not related?**

In [105]:
results = search(index, new_query)
results

[{'start': 0,
  'content': '## Evidently\n\n`Evidently` is available as a Python package. Install it using the **pip package manager**:\n\n```python\npip install evidently\n```\n\nTo install `evidently` using **conda installer**, run:\n\n```sh\nconda install -c conda-forge evidently\n```\n\n## Evidently LLM\n\nTo run evaluations specific to LLMs that include additional dependencies, run:\n\n```python\npip install evidently[llm]\n```\n\n## Tracely\n\nTo use tracing based on OpenTelemetry, install the sister package **tracely**:\n\n```sh\npip install tracely\n```',
  'title': 'Installation',
  'description': 'How to install the open-source Python library.',
  'icon': 'play',
  'filename': 'docs/setup/installation.mdx'},
 {'start': 0,
  'content': "You may need evaluations at different stages of your AI product development:\n\n* **Ad hoc analysis.** Spot-check the quality of your data or AI outputs.\n\n* **Experiments**. Test different parameters, models, or prompts and compare outcomes.\

In [103]:
new_query = "how do I run llm evals?"
structured_response = structured_rag(
    openai_client,
    prompt_template,
    index,
    new_query,
    AnswerResponse,
    model="gpt-5-mini-2025-08-07"
)
structured_response

AnswerResponse(answer_not_found=None, found_answer=True, answer=DocAssistantResponse(query='how do I run llm evals?', answer='High-level workflow\n\n1) Prepare your data and descriptors\n- Create an Evidently Dataset (you can build it from a pandas DataFrame). Use a DataDefinition to map your schema and compute descriptors (e.g., TextLength, SentenceCount or any LLM-specific descriptors). Descriptors are required for Text/LLM evaluations.\n\n2) Create a Report with the TextEvals preset\n- Use the TextEvals preset to summarize descriptor-level (LLM/text) metrics. You can optionally enable Tests (pass/fail conditions) by providing a reference dataset.\n\n3) Run the Report\n- Call report.run(current, ref) (or report.run(current, None) for a single dataset) to compute the evaluation. The result (my_eval) can be inspected in a notebook or serialized.\n\n4) (Optional) Upload to Evidently Cloud\n- If you use Evidently Cloud, connect with CloudWorkspace, create a Project, and upload the run wi